In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

DATA_DIR = Path("../../data/analysis_ready")
OUTPUT_ROOT = Path("../../outputs")
RESULT_DIR = OUTPUT_ROOT / "subgroup_analysis"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
COHORTS = {"emer": "Emergency", "elec": "Elective"}
OUTCOMES = {"30D": "30-day mortality", "60D": "60-day mortality", "90D": "90-day mortality", "surgco": "Surgery-related complication", "medco": "Medical-related complication"}
GROUPS = {
    "Age": (("Age <65 years", lambda frame: frame["age_group"].eq("<65")), ("Age >=65 years", lambda frame: frame["age_group"].eq(">=65"))),
    "ASA physical status": (("ASA 1-3", lambda frame: frame["asa_status"].isin([1, 2, 3])), ("ASA 4-5", lambda frame: frame["asa_status"].isin([4, 5]))),
}

def auc_with_ci(y_true, probability):
    auc = roc_auc_score(y_true, probability)
    events = int(np.sum(y_true))
    nonevents = len(y_true) - events
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    standard_error = np.sqrt((auc * (1 - auc) + (events - 1) * (q1 - auc * auc) + (nonevents - 1) * (q2 - auc * auc)) / (events * nonevents))
    lower, upper = np.clip([auc - 1.96 * standard_error, auc + 1.96 * standard_error], 0, 1)
    return auc, float(lower), float(upper)

rows = []
for cohort_code, cohort in COHORTS.items():
    membership = pd.read_csv(DATA_DIR / f"{cohort_code}_validation_subgroups.csv")
    for outcome_code, outcome in OUTCOMES.items():
        model_directory = OUTPUT_ROOT / "full_models" / f"{outcome_code}_{cohort_code}"
        for prediction_file in sorted(model_directory.glob("roc_data_*_V.csv")):
            prediction = pd.read_csv(prediction_file)
            if len(prediction) != len(membership):
                raise ValueError(f"Row mismatch between {prediction_file} and subgroup data")
            model = prediction_file.stem.replace("roc_data_", "").replace("_V", "")
            for subgroup_type, definitions in GROUPS.items():
                for subgroup, selector in definitions:
                    mask = selector(membership).to_numpy(bool)
                    y_true = prediction.loc[mask, "y_true"].to_numpy(int)
                    probability = prediction.loc[mask, "y_score"].to_numpy(float)
                    if len(np.unique(y_true)) < 2:
                        auc = lower = upper = np.nan
                    else:
                        auc, lower, upper = auc_with_ci(y_true, probability)
                    rows.append({"Subgroup_type": subgroup_type, "Subgroup": subgroup, "Cohort": cohort, "Outcome": outcome, "Model": model, "AUROC": auc, "AUROC_CI_low": lower, "AUROC_CI_high": upper})

results = pd.DataFrame(rows)
results.to_csv(RESULT_DIR / "subgroup_validation_auc_results.csv", index=False)

for subgroup_type, definitions in GROUPS.items():
    groups = [item[0] for item in definitions]
    figure, axes = plt.subplots(len(OUTCOMES), 2, figsize=(16, 18), sharex=True)
    for column, subgroup in enumerate(groups):
        for row, outcome in enumerate(OUTCOMES.values()):
            axis = axes[row, column]
            subset = results[(results["Subgroup_type"] == subgroup_type) & (results["Subgroup"] == subgroup) & (results["Outcome"] == outcome)]
            models = list(subset["Model"].drop_duplicates())
            for cohort_index, cohort in enumerate(COHORTS.values()):
                cohort_rows = subset[subset["Cohort"] == cohort].set_index("Model").reindex(models)
                position = np.arange(len(models)) + (-0.15 if cohort_index == 0 else 0.15)
                axis.errorbar(cohort_rows["AUROC"], position, xerr=[cohort_rows["AUROC"] - cohort_rows["AUROC_CI_low"], cohort_rows["AUROC_CI_high"] - cohort_rows["AUROC"]], fmt="o", label=cohort)
            axis.set_yticks(np.arange(len(models)), models)
            axis.invert_yaxis()
            axis.set_title(outcome)
            axis.axvline(0.5, linestyle="--", color="0.6")
            axis.grid(axis="x", alpha=0.2)
        axes[0, column].text(0.5, 1.18, subgroup, transform=axes[0, column].transAxes, ha="center", fontweight="bold")
    axes[0, 0].legend(frameon=False)
    figure.tight_layout()
    filename = "subgroup_auc_age.png" if subgroup_type == "Age" else "subgroup_auc_asa.png"
    figure.savefig(RESULT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.close(figure)
